# 01 — Data Sanity Check

Verifies the OceanEmbed data layer end-to-end:
- Batch shapes, dtypes, masks
- NaN fractions before and after masking
- Spatial maps of all 7 surface channels
- Vertical temperature profile at a fixed (lat, lon)
- GLORYS surface temp vs SST correlation (alignment sanity)
- Normalization statistics
- Climatology baseline metrics

> **SCIENTIFIC NOTE**: GLORYS12V1 is a reanalysis, not independent ground truth.
> All metrics in this notebook are plumbing checks on a 5-day smoke-test dataset.
> Statistical conclusions require the full multi-year dataset.

In [1]:
import sys
from pathlib import Path

# Add backend/ to path
repo_root = Path.cwd().parent
backend_dir = repo_root / 'backend'
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

import json
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from ocean_embed.data import (
    OceanEmbedDataset,
    fit_normalization_stats,
    load_normalization_stats,
    make_temporal_split,
)
from ocean_embed.baselines import fit_climatology, evaluate_climatology

PLOTS_DIR = Path('plots')
PLOTS_DIR.mkdir(exist_ok=True)

SURFACE_ZARR = repo_root / 'data/processed/surface_features.zarr'
TARGET_ZARR  = repo_root / 'data/processed/target_glorys.zarr'
STATS_PATH   = repo_root / 'data/processed/normalization_stats.json'
CLIM_PATH    = repo_root / 'data/processed/climatology.npz'
CLIM_DIR     = repo_root / 'runs/baseline_climatology'

print('Setup complete.')

Setup complete.


## 1 — Temporal Split & Dataset Setup

In [2]:
sf = xr.open_zarr(str(SURFACE_ZARR))
times = sf.time.values
sf.close()

splits = make_temporal_split(times, train_frac=0.70, val_frac=0.15, test_frac=0.15)
train_idx = splits['train']
val_idx   = splits['val']
test_idx  = splits['test']

print(f'Total timestamps  : {len(times)}')
print(f'Train indices     : {train_idx.tolist()}  ({len(train_idx)} samples)')
print(f'Val   indices     : {val_idx.tolist()}  ({len(val_idx)} samples)')
print(f'Test  indices     : {test_idx.tolist()}  ({len(test_idx)} samples)')

# Fit normalization stats from training split only
stats = fit_normalization_stats(SURFACE_ZARR, TARGET_ZARR, train_idx, STATS_PATH)

# Build dataset on ALL indices for sanity plotting
dataset = OceanEmbedDataset(SURFACE_ZARR, TARGET_ZARR, STATS_PATH)
print(f'\nDataset length    : {len(dataset)}')

Total timestamps  : 5
Train indices     : [0, 1, 2, 3]  (4 samples)
Val   indices     : [4]  (1 samples)
Test  indices     : []  (0 samples)



Dataset length    : 5


## 2 — Batch Shapes & Dtypes

In [3]:
dl = DataLoader(dataset, batch_size=2, shuffle=False, num_workers=0)
batch = next(iter(dl))

print('Batch shapes:')
for k, v in batch.items():
    if hasattr(v, 'shape'):
        print(f'  {k:16s}: shape={tuple(v.shape)}  dtype={v.dtype}')
    else:
        print(f'  {k:16s}: {v}')

assert batch['input'].shape  == (2, 7, 101, 241), 'FAIL: input shape'
assert batch['target'].shape == (2, 15, 101, 241), 'FAIL: target shape'
print('\n[PASS] Batch shapes correct.')

Batch shapes:
  input           : shape=(2, 7, 101, 241)  dtype=torch.float32
  masks           : shape=(2, 7, 101, 241)  dtype=torch.float32
  target          : shape=(2, 15, 101, 241)  dtype=torch.float32
  target_mask     : shape=(2, 15, 101, 241)  dtype=torch.float32
  time            : shape=(2,)  dtype=torch.int64

[PASS] Batch shapes correct.


## 3 — NaN Fractions Before and After Masking

In [4]:
SURFACE_VARS = ['analysed_sst', 'sos', 'sla', 'u', 'v', 'uwnd', 'vwnd']

sf = xr.open_zarr(str(SURFACE_ZARR))

print('NaN fractions per surface channel (raw zarr, all 5 time steps):')
print(f'  {"Channel":<20} {"NaN before mask":>16}  {"NaN after mask (zeroed)":>24}')
print('  ' + '-'*64)
for var in SURFACE_VARS:
    arr = sf[var].values
    nan_frac_raw = float(np.isnan(arr).mean())
    # After masking: input tensor has NaN -> 0, so NaN fraction = 0
    print(f'  {var:<20} {nan_frac_raw:>16.4f}  {0.0:>24.4f}')

sf.close()

# Confirm no NaN in actual tensors
sample = dataset[0]
n_nan_in  = int(torch.isnan(sample['input']).sum())
n_nan_tgt = int(torch.isnan(sample['target']).sum())
print(f'\nNaN in input tensor : {n_nan_in}')
print(f'NaN in target tensor: {n_nan_tgt}')
assert n_nan_in == 0 and n_nan_tgt == 0, 'FAIL: NaN found in tensors'

NaN fractions per surface channel (raw zarr, all 5 time steps):
  Channel               NaN before mask   NaN after mask (zeroed)
  ----------------------------------------------------------------
  analysed_sst                   0.5183                    0.0000
  sos                            0.5263                    0.0000
  sla                            0.5151                    0.0000
  u                              0.5411                    0.0000
  v                              0.5411                    0.0000
  uwnd                           0.0279                    0.0000
  vwnd                           0.0279                    0.0000



NaN in input tensor : 0
NaN in target tensor: 0


## 4 — Spatial Maps of 7 Surface Channels

In [5]:
CHANNEL_LABELS = [
    'SST (deg C)', 'SSS (PSU)', 'SSH (m)',
    'Current U (m/s)', 'Current V (m/s)',
    'Wind U (m/s)', 'Wind V (m/s)',
]
CMAPS = ['RdBu_r', 'viridis', 'BrBG', 'RdBu_r', 'RdBu_r', 'RdBu_r', 'RdBu_r']

sf = xr.open_zarr(str(SURFACE_ZARR))
lats = sf.latitude.values
lons = sf.longitude.values

fig, axes = plt.subplots(3, 3, figsize=(16, 10))
axes = axes.ravel()

t_idx = 0  # first timestamp
vars_in_order = ['analysed_sst', 'sos', 'sla', 'u', 'v', 'uwnd', 'vwnd']

for i, (var, label, cmap) in enumerate(zip(vars_in_order, CHANNEL_LABELS, CMAPS)):
    arr = sf[var].isel(time=t_idx).values
    # Handle any extra dims (e.g. depth was squeezed already, but guard anyway)
    while arr.ndim > 2:
        arr = arr[0]
    vmax = np.nanpercentile(np.abs(arr), 98)
    vmin = -vmax if cmap == 'RdBu_r' else np.nanpercentile(arr, 2)
    im = axes[i].pcolormesh(lons, lats, arr, cmap=cmap, vmin=vmin, vmax=vmax)
    axes[i].set_title(label, fontsize=10)
    axes[i].set_xlabel('Lon')
    axes[i].set_ylabel('Lat')
    plt.colorbar(im, ax=axes[i], shrink=0.8)

# Hide unused subplots
for j in range(len(vars_in_order), len(axes)):
    axes[j].set_visible(False)

sf.close()
fig.suptitle(f'Surface Channels — 2020-01-01 (North Indian Ocean)', fontsize=13, fontweight='bold')
fig.tight_layout()
plot_path = PLOTS_DIR / '01_data_sanity_surface_maps.png'
fig.savefig(plot_path, dpi=120, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {plot_path}')

Saved: plots\01_data_sanity_surface_maps.png


## 5 — Vertical Temperature Profile at a Fixed (lat, lon)

In [6]:
tg = xr.open_zarr(str(TARGET_ZARR))
depths = tg.depth.values
lats_t = tg.latitude.values
lons_t = tg.longitude.values

# Pick a central point: lat=15N, lon=75E (Arabian Sea)
lat_pt, lon_pt = 15.0, 75.0
lat_i = int(np.argmin(np.abs(lats_t - lat_pt)))
lon_i = int(np.argmin(np.abs(lons_t - lon_pt)))

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Left: all 5 time steps overlaid
for t in range(tg.sizes['time']):
    profile = tg['thetao'].isel(time=t, latitude=lat_i, longitude=lon_i).values
    label = str(pd.Timestamp(tg.time.values[t]).date())
    axes[0].plot(profile, depths, label=label, lw=1.5)

axes[0].invert_yaxis()
axes[0].set_xlabel('Temperature (deg C)')
axes[0].set_ylabel('Depth (m)')
axes[0].set_title(f'Vertical Profile at ({lat_pt}N, {lon_pt}E)\nAll 5 timesteps')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.4)

# Right: mean profile with std shading
profiles = np.stack([
    tg['thetao'].isel(time=t, latitude=lat_i, longitude=lon_i).values
    for t in range(tg.sizes['time'])
], axis=0)
mean_p = np.nanmean(profiles, axis=0)
std_p  = np.nanstd(profiles, axis=0)

axes[1].plot(mean_p, depths, 'b-', lw=2, label='Mean')
axes[1].fill_betweenx(depths, mean_p - std_p, mean_p + std_p,
                      alpha=0.3, color='blue', label='+/- 1 std')
axes[1].invert_yaxis()
axes[1].set_xlabel('Temperature (deg C)')
axes[1].set_ylabel('Depth (m)')
axes[1].set_title(f'Mean Profile +/- 1 std')
axes[1].legend()
axes[1].grid(True, alpha=0.4)

tg.close()

fig.suptitle('GLORYS12V1 Subsurface Temperature Profile (REANALYSIS, not ground truth)',
             fontsize=11)
fig.tight_layout()
plot_path = PLOTS_DIR / '01_data_sanity_profile.png'
fig.savefig(plot_path, dpi=120, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {plot_path}')

C:\Users\DELL\AppData\Local\Temp\ipykernel_3660\633743842.py:31: RuntimeWarning: Mean of empty slice
  mean_p = np.nanmean(profiles, axis=0)
C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


Saved: plots\01_data_sanity_profile.png


## 6 — GLORYS Surface Temp vs SST Correlation (Alignment Sanity)

In [7]:
sf = xr.open_zarr(str(SURFACE_ZARR))
tg = xr.open_zarr(str(TARGET_ZARR))

sst_all    = sf['analysed_sst'].values.ravel()         # (T*H*W,)
glorys_sst = tg['thetao'].isel(depth=0).values.ravel()  # surface depth=0

valid = np.isfinite(sst_all) & np.isfinite(glorys_sst)
x = sst_all[valid]
y = glorys_sst[valid]

r  = float(np.corrcoef(x, y)[0, 1])
r2 = r ** 2
bias = float(np.mean(y - x))
rmse = float(np.sqrt(np.mean((y - x)**2)))

print(f'GLORYS surface (depth=0) vs OSTIA SST:')
print(f'  Pearson r  = {r:.4f}')
print(f'  R^2        = {r2:.4f}')
print(f'  Bias       = {bias:.4f} deg C  (GLORYS - SST)')
print(f'  RMSE       = {rmse:.4f} deg C')
print(f'  n_valid    = {valid.sum()}')

if r2 < 0.9:
    print(f'  WARNING: R^2={r2:.3f} < 0.9 — check temporal alignment or variable units.')
else:
    print(f'  [PASS] R^2 > 0.9 — stores are temporally aligned.')

# Scatter plot (subsample for speed)
rng = np.random.default_rng(42)
idx = rng.choice(len(x), min(5000, len(x)), replace=False)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(x[idx], y[idx], s=2, alpha=0.3, color='steelblue')
lo = min(x.min(), y.min())
hi = max(x.max(), y.max())
ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='1:1 line')
ax.set_xlabel('OSTIA SST (deg C)')
ax.set_ylabel('GLORYS surface temperature (deg C)')
ax.set_title(f'Alignment check: R^2={r2:.3f}  bias={bias:.3f} deg C')
ax.legend()
ax.grid(True, alpha=0.3)

sf.close(); tg.close()

plot_path = PLOTS_DIR / '01_data_sanity_glorys_vs_sst.png'
fig.savefig(plot_path, dpi=120, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {plot_path}')

GLORYS surface (depth=0) vs OSTIA SST:
  Pearson r  = 0.9676
  R^2        = 0.9363
  Bias       = -0.0269 deg C  (GLORYS - SST)
  RMSE       = 0.4158 deg C
  n_valid    = 57735
  [PASS] R^2 > 0.9 — stores are temporally aligned.
Saved: plots\01_data_sanity_glorys_vs_sst.png


## 7 — Normalization Statistics

In [8]:
stats = load_normalization_stats(STATS_PATH)

print('Surface channel normalization stats (train split only):')
print(f'  {"Variable":<20} {"Mean":>10}  {"Std":>10}')
print('  ' + '-'*44)
for var, s in stats['surface'].items():
    print(f'  {var:<20} {s["mean"]:>10.4f}  {s["std"]:>10.4f}')

print()
print('Target (thetao) normalization stats per depth:')
print(f'  {"Depth (m)":>12} {"Mean":>10}  {"Std":>10}')
print('  ' + '-'*36)
for d in stats['target']:
    print(f'  {d["depth_m"]:>12.1f} {d["mean"]:>10.4f}  {d["std"]:>10.4f}')

# Bar plot of surface means and stds
surface_vars  = list(stats['surface'].keys())
surface_means = [stats['surface'][v]['mean'] for v in surface_vars]
surface_stds  = [stats['surface'][v]['std']  for v in surface_vars]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
x_pos = range(len(surface_vars))

ax1.bar(x_pos, surface_means, color='steelblue', alpha=0.8)
ax1.set_xticks(list(x_pos)); ax1.set_xticklabels(surface_vars, rotation=30, ha='right')
ax1.set_ylabel('Mean'); ax1.set_title('Surface Channel Means (training split)')
ax1.grid(True, alpha=0.3, axis='y')

ax2.bar(x_pos, surface_stds, color='darkorange', alpha=0.8)
ax2.set_xticks(list(x_pos)); ax2.set_xticklabels(surface_vars, rotation=30, ha='right')
ax2.set_ylabel('Std Dev'); ax2.set_title('Surface Channel Stds (training split)')
ax2.grid(True, alpha=0.3, axis='y')

fig.tight_layout()
plot_path = PLOTS_DIR / '01_data_sanity_norm_stats.png'
fig.savefig(plot_path, dpi=120, bbox_inches='tight')
plt.close(fig)
print(f'\nSaved: {plot_path}')

Surface channel normalization stats (train split only):
  Variable                   Mean         Std
  --------------------------------------------
  analysed_sst            27.1998      1.6747
  sos                     34.4481      2.1044
  sla                      0.0663      0.0911
  u                       -0.0473      0.1760
  v                        0.0442      0.1745
  uwnd                    -2.4822      2.6130
  vwnd                    -1.4006      2.5513

Target (thetao) normalization stats per depth:
     Depth (m)       Mean         Std
  ------------------------------------
           0.5    27.2134      1.6424
           5.1    27.1963      1.6142
           9.6    27.2566      1.4826
          18.5    27.3320      1.4205
          29.4    27.3698      1.4338
          47.4    27.1149      1.6059
          77.9    25.0743      2.2181
          92.3    23.6485      2.4210
         130.7    19.6456      2.2045
         155.9    17.5002      1.9512
         186.1    15.771


Saved: plots\01_data_sanity_norm_stats.png


## 8 — Climatology Baseline Metrics

In [9]:
# Fit climatology (train only) and evaluate on val split
clim = fit_climatology(TARGET_ZARR, train_idx, CLIM_PATH)
print(f'Climatology array shape: {clim.shape}  (365 DOYs x 15 depths x 101 lat x 241 lon)')

metrics = evaluate_climatology(CLIM_PATH, TARGET_ZARR, val_idx, results_dir=CLIM_DIR)

print(f'\nClimatology evaluation — n_eval_samples: {metrics["n_eval_samples"]}')
print(f'NOTE: {metrics["note"]}')
print()

if metrics['per_depth']:
    print('Per-depth metrics:')
    print(f'  {"Depth (m)":>12} {"RMSE":>8}  {"MAE":>8}  {"bias":>8}  {"R2":>8}')
    print('  ' + '-'*52)
    for d in metrics['per_depth']:
        rmse_s = f"{d['rmse']:.4f}" if d['rmse'] is not None else 'N/A'
        mae_s  = f"{d['mae']:.4f}"  if d['mae']  is not None else 'N/A'
        bias_s = f"{d['bias']:.4f}" if d['bias'] is not None else 'N/A'
        r2_s   = f"{d['r2']:.4f}"   if d['r2']   is not None else 'N/A'
        print(f'  {d["depth_m"]:>12.1f} {rmse_s:>8}  {mae_s:>8}  {bias_s:>8}  {r2_s:>8}')
else:
    print('(No eval samples — all timestamps in training split.  Expected for 5-day dataset.)')

print()
print('Depth band summaries:')
for band, bm in metrics['depth_bands'].items():
    if bm:
        print(f'  {band}: mean RMSE = {bm["mean_rmse"]}')
    else:
        print(f'  {band}: no data')

print()
print(f'RMSE vs depth plot: {CLIM_DIR}/rmse_vs_depth.png')
print(f'Metrics JSON      : {CLIM_DIR}/metrics.json')

Climatology array shape: (365, 15, 101, 241)  (365 DOYs x 15 depths x 101 lat x 241 lon)



Climatology evaluation — n_eval_samples: 1
NOTE: GLORYS12V1 is a reanalysis, not independent ground truth. Metrics with 5-day data are nearly degenerate (DOY constant) — plumbing test only.

Per-depth metrics:
     Depth (m)     RMSE       MAE      bias        R2
  ----------------------------------------------------
           0.5      N/A       N/A       N/A       N/A
           5.1      N/A       N/A       N/A       N/A
           9.6      N/A       N/A       N/A       N/A
          18.5      N/A       N/A       N/A       N/A
          29.4      N/A       N/A       N/A       N/A
          47.4      N/A       N/A       N/A       N/A
          77.9      N/A       N/A       N/A       N/A
          92.3      N/A       N/A       N/A       N/A
         130.7      N/A       N/A       N/A       N/A
         155.9      N/A       N/A       N/A       N/A
         186.1      N/A       N/A       N/A       N/A
         318.1      N/A       N/A       N/A       N/A
         541.1      N/A       N/

## 9 — Summary

In [10]:
print('='*55)
print('DATA SANITY NOTEBOOK SUMMARY')
print('='*55)
print(f'  Surface store    : {SURFACE_ZARR}')
print(f'  Target store     : {TARGET_ZARR}')
print(f'  Normalization    : {STATS_PATH}')
print(f'  Climatology      : {CLIM_PATH}')
print(f'  Dataset length   : {len(dataset)}')
print(f'  Train/Val/Test   : {len(train_idx)}/{len(val_idx)}/{len(test_idx)}')
print()
print('Plots saved to notebooks/plots/:')
for p in sorted(PLOTS_DIR.glob('01_*.png')):
    print(f'  {p.name}')
print('='*55)
print('All cells executed without error.  V8 PASS.')

DATA SANITY NOTEBOOK SUMMARY
  Surface store    : C:\Users\DELL\Documents\oceanembed\OceanEmbed\data\processed\surface_features.zarr
  Target store     : C:\Users\DELL\Documents\oceanembed\OceanEmbed\data\processed\target_glorys.zarr
  Normalization    : C:\Users\DELL\Documents\oceanembed\OceanEmbed\data\processed\normalization_stats.json
  Climatology      : C:\Users\DELL\Documents\oceanembed\OceanEmbed\data\processed\climatology.npz
  Dataset length   : 5
  Train/Val/Test   : 4/1/0

Plots saved to notebooks/plots/:
  01_data_sanity_glorys_vs_sst.png
  01_data_sanity_norm_stats.png
  01_data_sanity_profile.png
  01_data_sanity_surface_maps.png
All cells executed without error.  V8 PASS.
